In [5]:
import sys, os
sys.path.append(os.path.abspath('../../src'))

from doc_loader import show_doc


# Data Export


---

## 4. Merge mit Tram-IST-Daten

Um den Master-Wetter-Datensatz mit den Tram-IST-Daten zu verknüpfen:

1. **Tram-Zeitstempel runden:** Sekundengenaue IST-Zeitstempel werden auf die volle Stunde abgerundet, sodass sie dem `datum`-Schlüssel des Wetter-Datensatzes entsprechen.

2. **Left Join:** Die Tram-IST-Daten werden als linke Tabelle geführt — jede Fahrt erhält die Wetterdaten der entsprechenden Stunde.

   ```python
   tram_df['datum'] = tram_df['timestamp'].dt.floor('h')
   merged = tram_df.merge(meteo_df, on='datum', how='left')
   ```

3. **EDA-Themen (nach dem Merge):**
   - Schwellenwert für `flood_alert` (Binarisierung von `flood_intensity`) bestimmen.
   - Niederschlagskategorien (`kein`, `leicht`, `stark`) aus `precipitation_mm` ableiten.
   - Korrelation Temperatur / Regen / Schnee vs. Verspätung untersuchen.
   - Schnee - **`is_snow`**: `True`, wenn `T < 1.0°C` **UND** `RainDur > 0`. Beide Werte stammen von der Referenz-Station Stampfenbachstrasse. Identifiziert Schneefallphasen, die den Trambetrieb erfahrungsgemäß stark verlangsamen.



In [6]:
show_doc('events')

## Datenwörterbuch — Events-Master-Dataset

| Spalte | Dtype | Beschreibung |
| :--- | :--- | :--- |
| **Datum** | datetime64[us] | Datum des Events (tagesgenau, keine Uhrzeit). Join-Schlüssel für Tram-IST-Daten über `dt.normalize()`. |
| **Event_Name** | str | Bezeichnung des Events. |
| **Typ** | str | Kategorie: Feiertag, Stadtfest, Konzert, Fachmesse, Kongress, Super League, Schweizer Cup, UEFA … |
| **Gewichtung** | float64 | Besucherintensität: 1 = Mittel (1k–10k), 2 = Hoch (10k–30k), 3 = Sehr hoch (>30k). |
| **Ort** | str | Veranstaltungsort / Stadtteil. |


> **EDA-Aufgaben:**    
* Binäres Feature `has_event` (0/1),    
* Kombination von Gewichtung und Typ als kategoriale Variable, Analyse Verspätung vs. Gewichtungsstufe.   



In [7]:
show_doc('meteo')


## Datenwörterbuch — Meteo-Master-Dataset

| Spalte | Dtype | Einheit | Quelle | Beschreibung |
| :--- | :--- | :--- | :--- | :--- |
| **date_time** | datetime64[us] | — | UGZ | Zeitstempel, stündlich, CET, tz-naiv |
| **temperature** | float64 | °C | UGZ Stampfenbachstrasse | Lufttemperatur. Basis für `is_snow` in der EDA. |
| **humidity** | float64 | %Hr | UGZ Stampfenbachstrasse | Relative Luftfeuchtigkeit. |
| **air_pressure** | float64 | hPa | UGZ Stampfenbachstrasse | Luftdruck. |
| **rain_duration** | float64 | min | UGZ Stampfenbachstrasse | Niederschlagsdauer pro Stunde. |
| **global_radiation** | float64 | W/m² | UGZ Stampfenbachstrasse | Globalstrahlung (Sonneneinstrahlung). |
| **wind_direction** | float64 | ° | UGZ Stampfenbachstrasse | Windrichtung (0–360°). |
| **wind_speed** | float64 | m/s | UGZ Stampfenbachstrasse | Windgeschwindigkeit (skalar). |
| **wind_speed_vector** | float64 | m/s | UGZ Stampfenbachstrasse | Windgeschwindigkeit (Vektor). |
| **precipitation_mm** | float64 | mm | Wapo Mythenquai | Niederschlagsmenge, stündlich summiert. NaN wenn keine Wapo-Messung. |
| **flood_intensity** | int64 | Anzahl | ERZ | Summe Überschwemmungsmeldungen des Tages über alle Zonen. 0 = kein Ereignis. |

> **EDA-Aufgaben:**   
* `is_snow` (temperature < 1°C & rain_duration > 0),   
* `flood_alert` (Schwellenwert aus Verteilung),   
* Regen-Kategorien aus `precipitation_mm`.  